# PDZ CIS — Exponential-Exponential Model: Initial Data Exploration

This notebook analyzes the Bayesian model fit for the PDZ cis DMS data. We look at the HMC/MLE result artifacts produced by Bruce using SciStanPy (FLIP branch), and is organized to more or less mirror the `flipv3/flip_data_analysis.py` workflow.

## Experiment Overview

**Dataset**: Real deep mutational scanning (DMS) data for PDZ domain proteins  
**Selection regime**: *Cis*  
**Model family in FLIP**: PDZ3 enrichment models  
**Current run**: exponential growth curve + exponential growth-rate prior (the PDZEgEr configuration)

## Data Files

Two inference results are stored in NetCDF4/ArviZ format:

| File | Size | Method | Contents |
|------|------|--------|----------|
| `pdz_cis_exponential_exponential_diagnosed.nc` | ~9.5 GB | HMC | Posterior samples + `sample_stats` diagnostics |
| `pdz_cis_exponential_exponential_samples.nc` | ~530 MB | MLE | Point estimate + bootstrap samples |

## Precise Model Pointers (PDZ)

For PDZ-specific model definitions and exact configuration logic, use:

- **PDZ model file**: [flipv3/pdz3_models.py](https://github.com/microsoft/SciStanPy/blob/flip/flipv3/pdz3_models.py)
- **Base PDZ class**: `BasePDZ3` (inherits enrichment template and sets PDZ-specific initialization)
- **Model-class mapping**: `PDZ3_MODELS` dictionary
- **Model selector**: `get_pdz3_model(growth_curve, growth_rate)`
- **PDZ instance factory**: `get_pdz3_instance(filepath, growth_curve, growth_rate, **hyperparams)`
- **Exponential-exponential model key**: `("exponential", "exponential") -> PDZEgEr`

## Analysis Workflow Pointers

The main analysis file is the one shared by Bruce over email, [flipv3/flip_data_analysis.py](https://github.com/microsoft/SciStanPy/blob/flip/flipv3/flip_data_analysis.py). here we try to replicate some of the functions, in particular:
  - `reshape_fitness(...)`
  - `get_fitness_correlations(...)`
  - `run_analysis(...)` (HMC/MLE parallel summaries)

## More General Infrastructure References
- **SciStanPy repo** (`flip` branch): [github.com/microsoft/SciStanPy](https://github.com/microsoft/SciStanPy/tree/flip)
- **Pipelines used to run jobs**: [scistanpy/pipelines](https://github.com/microsoft/SciStanPy/tree/flip/scistanpy/pipelines)
- **SciStanPy docs**: [microsoft.github.io/SciStanPy](https://microsoft.github.io/SciStanPy/)

In [1]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import warnings
from pathlib import Path

import holoviews as hv
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats

from scistanpy.model.results import MLEInferenceRes, SampleResults
from scistanpy.plotting import quantile_plot
from scistanpy.utils import faster_autocorrelation

hv.extension("bokeh")

In [2]:
# ── 2. Path configuration (HMC + MLE) ────────────────────────────────────────
from pathlib import Path

# Files are stored in the repo's data/ folder
repo_root = Path("..").resolve()  # navigate up from notebooks/
data_dir = repo_root / "data"

hmc_file = data_dir / "pdz_cis_exponential_exponential_diagnosed.nc"
mle_file = data_dir / "pdz_cis_exponential_exponential_samples.nc"

assert hmc_file.exists(), f"HMC file not found: {hmc_file}"
assert mle_file.exists(), f"MLE file not found: {mle_file}"

hmc_size_mb = hmc_file.stat().st_size / 1e6
mle_size_mb = mle_file.stat().st_size / 1e6

print(f"HMC file: {hmc_file.name}  ({hmc_size_mb:.1f} MB)")
print(f"MLE file: {mle_file.name}  ({mle_size_mb:.1f} MB)")

HMC file: pdz_cis_exponential_exponential_diagnosed.nc  (9913.8 MB)
MLE file: pdz_cis_exponential_exponential_samples.nc  (555.5 MB)


## Load Both NetCDF Results (HMC and MLE)

We load both files up front so we can compute and compare results in separate sections.

- **HMC** (`pdz_cis_exponential_exponential_diagnosed.nc`): posterior draws + MCMC diagnostics.
- **MLE** (`pdz_cis_exponential_exponential_samples.nc`): point estimate + bootstrap samples.

In [3]:
# ── 3. Load both results ───────────────────────────────────
idata_hmc = None
idata_mle = None
hmc_samples = None
mle_samples = None

# HMC result object (contains InferenceData at .inference_obj)
hmc_samples = SampleResults.from_disk(str(hmc_file), skip_fit=True)
idata_hmc = hmc_samples.inference_obj

# MLE/Bootstrap result object (contains InferenceData at .inference_obj)
mle_samples = MLEInferenceRes.from_disk(str(mle_file))
idata_mle = mle_samples.inference_obj

print("Loaded HMC and MLE via SciStanPy result classes.")

Loaded HMC and MLE via SciStanPy result classes.


c:\Users\t-renatob\OneDrive - Microsoft\Desktop\RenatoBerlinghieri\prob-protein-fitness\.venv312\Lib\site-packages\arviz\data\inference_data.py:157: UserWarning: variable_diagnostic_stats group is not defined in the InferenceData scheme
  warnings.warn(
c:\Users\t-renatob\OneDrive - Microsoft\Desktop\RenatoBerlinghieri\prob-protein-fitness\.venv312\Lib\site-packages\arviz\data\inference_data.py:157: UserWarning: sample_diagnostic_tests group is not defined in the InferenceData scheme
  warnings.warn(
c:\Users\t-renatob\OneDrive - Microsoft\Desktop\RenatoBerlinghieri\prob-protein-fitness\.venv312\Lib\site-packages\arviz\data\inference_data.py:157: UserWarning: variable_diagnostic_tests group is not defined in the InferenceData scheme
  warnings.warn(
c:\Users\t-renatob\OneDrive - Microsoft\Desktop\RenatoBerlinghieri\prob-protein-fitness\.venv312\Lib\site-packages\arviz\data\inference_data.py:157: UserWarning: mle group is not defined in the InferenceData scheme
  warnings.warn(


## Part A — HMC Results

This section loads/inspects HMC posterior groups and diagnostics (`sample_stats`) from the diagnosed file.

In [4]:
# ── 4A. HMC structural overview ───────────────────────────────────────────────
print("=== HMC InferenceData groups ===")
for group in idata_hmc.groups():
    grp_ds = getattr(idata_hmc, group)
    print(f"\n[{group}] dims={dict(grp_ds.sizes)}")
    for var in list(grp_ds.data_vars):
        arr = grp_ds[var]
        dims_text = ", ".join(f"{dim}={arr.sizes[dim]}" for dim in arr.dims)
        print(f"  {var:<35s} dims=({dims_text}) dtype={arr.dtype}")

=== HMC InferenceData groups ===

[posterior] dims={'chain': 8, 'draw': 2000, 'a': 3, 'b': 10660}
  log_theta_t0                        dims=(chain=8, draw=2000, a=3, b=10660) dtype=float32
  log_r_mean                          dims=(chain=8, draw=2000, b=10660) dtype=float32
  r                                   dims=(chain=8, draw=2000, a=3, b=10660) dtype=float32
  raw_abundances_tg0                  dims=(chain=8, draw=2000, a=3, b=10660) dtype=float32
  log_theta_tg0                       dims=(chain=8, draw=2000, a=3, b=10660) dtype=float32

[posterior_predictive] dims={'chain': 8, 'draw': 2000, 'a': 3, 'b': 10660}
  starting_counts                     dims=(chain=8, draw=2000, a=3, b=10660) dtype=int32
  timepoint_counts                    dims=(chain=8, draw=2000, a=3, b=10660) dtype=int32

[sample_stats] dims={'chain': 8, 'draw': 2000}
  lp__                                dims=(chain=8, draw=2000) dtype=float32
  accept_stat__                       dims=(chain=8, draw=2000) d

## Part B — MLE Results

This section inspects the MLE/Bootstrap file and summarizes the bootstrap-based uncertainty quantities.

In [4]:
# ── 5B. MLE key variable inspection ───────────────────────────────────────────
print("=== MLE/Bootstrap groups ===")
for group in idata_mle.groups():
    grp = getattr(idata_mle, group)
    print(f"\n{'='*65}")
    print(f"  Group: {group}")
    print(f"  dims: {dict(grp.sizes)}")
    print(f"{'='*65}")
    for var in list(grp.data_vars):
        arr = grp[var]
        vals = arr.values.flatten().astype(float)
        finite = vals[np.isfinite(vals)]
        dims_text = ", ".join(f"{dim}={arr.sizes[dim]}" for dim in arr.dims)
        if finite.size > 0:
            print(
                f"  {var:<30s} dims=({dims_text}) "
                f"mean={finite.mean():.4g} std={finite.std():.4g}"
            )
        else:
            print(f"  {var:<30s} dims=({dims_text}) (no finite values)")

=== MLE/Bootstrap groups ===

  Group: posterior
  dims: {'chain': 1, 'draw': 1000, 'a': 3, 'b': 10660}
  log_theta_t0                   dims=(chain=1, draw=1000, a=3, b=10660) mean=-9.378 std=0.4705
  log_r_mean                     dims=(chain=1, draw=1000, b=10660) mean=-2.88 std=1.283
  r                              dims=(chain=1, draw=1000, a=3, b=10660) mean=1.286 std=1.28
  raw_abundances_tg0             dims=(chain=1, draw=1000, a=3, b=10660) mean=0 std=0
  log_theta_tg0                  dims=(chain=1, draw=1000, a=3, b=10660) mean=0 std=0

  Group: posterior_predictive
  dims: {'chain': 1, 'draw': 1000, 'a': 3, 'b': 10660}
  starting_counts                dims=(chain=1, draw=1000, a=3, b=10660) mean=628.6 std=984
  timepoint_counts               dims=(chain=1, draw=1000, a=3, b=10660) mean=631.7 std=2490

  Group: observed_data
  dims: {'a': 3, 'b': 10660}
  starting_counts                dims=(a=3, b=10660) mean=628.6 std=1006
  timepoint_counts               dims=(a=3, b=106

In [5]:
idata_mle.posterior["log_r_mean"]

<xarray.DataArray 'log_r_mean' (chain: 1, draw: 1000, b: 10660)> Size: 43MB
array([[[-2.475883, -4.05493 , ..., -2.367854, -1.026621],
        [-5.885267, -1.94378 , ..., -3.080994, -2.058864],
        ...,
        [-1.535924, -1.351595, ..., -1.849099, -4.781302],
        [-3.306297, -0.427245, ..., -3.333521, -4.201428]]],
      shape=(1, 1000, 10660), dtype=float32)
Dimensions without coordinates: chain, draw, b

## Combined Analysis

This section mirrors the structure of `flip_data_analysis.py` as closely as possible using the two NetCDF files currently available.

The original starting and timepoint counts are available in the MLE result file under `idata_mle.observed_data.starting_counts` and `idata_mle.observed_data.timepoint_counts`. What is still external to these result files is the raw FLIP metadata used by the original script, especially reported fitness values and variant labels from `flip_data_loc`.

In [6]:
# ── 6. Combined analysis (mirrors flip_data_analysis.py) ─────────────────────
# Runtime guards for exploratory use. Set max_resamples=None to use all HMC resamples.
max_resamples = 1000
build_plots = False
sample_dim = "resample"
variant_axis = "a"

reported_starting_counts = idata_mle.observed_data.starting_counts
reported_timepoint_counts = idata_mle.observed_data.timepoint_counts
variant_dim = "b" if "b" in reported_starting_counts.dims else reported_starting_counts.dims[-1]
variants = np.arange(reported_starting_counts.sizes[variant_dim])

# The NetCDF files include the raw observed counts, but not the independent
# reported-fitness CSV used by flip_data_analysis.py for its sanity check.
reported_fitness_values = None

# The original run_analysis expects get_growth_rate("mle") to be 1D. In this
# result, log_r_mean is the per-variant growth-rate summary in both MLE and HMC;
# r still carries the replicate/library axis and is therefore not the same object.
growth_rate_variable = "log_r_mean"
if growth_rate_variable not in idata_mle.mle:
    raise KeyError(f"{growth_rate_variable!r} not found in idata_mle.mle")
if growth_rate_variable not in idata_hmc.posterior:
    raise KeyError(f"{growth_rate_variable!r} not found in idata_hmc.posterior")
if idata_mle.mle[growth_rate_variable].dims != (variant_dim,):
    raise ValueError(
        f"Expected MLE {growth_rate_variable!r} to be 1D over {variant_dim!r}; "
        f"found dims={idata_mle.mle[growth_rate_variable].dims}"
    )
if not {"chain", "draw", variant_dim}.issubset(idata_hmc.posterior[growth_rate_variable].dims):
    raise ValueError(
        f"Expected HMC {growth_rate_variable!r} to contain chain/draw/{variant_dim!r}; "
        f"found dims={idata_hmc.posterior[growth_rate_variable].dims}"
    )


def _to_resamples(arr: xr.DataArray, max_resamples: int | None = None) -> xr.DataArray:
    """Collapse independent sample dimensions into one resample axis."""
    if {"chain", "draw"}.issubset(arr.dims):
        arr = arr.stack({sample_dim: ("chain", "draw")})
    elif "draw" in arr.dims:
        arr = arr.rename({"draw": sample_dim})

    if max_resamples is not None and sample_dim in arr.dims and arr.sizes[sample_dim] > max_resamples:
        idx = np.linspace(0, arr.sizes[sample_dim] - 1, max_resamples).astype(int)
        arr = arr.isel({sample_dim: idx})

    return arr


def _as_variant_array(arr: xr.DataArray) -> xr.DataArray:
    """Return an array with variant dimension named 'a', matching reshape_fitness."""
    if variant_axis in arr.dims and variant_dim not in arr.dims:
        return arr
    if variant_dim in arr.dims:
        other_feature_dims = [
            dim for dim in arr.dims if dim not in {sample_dim, variant_dim}
        ]
        if other_feature_dims:
            arr = arr.mean(dim=other_feature_dims, skipna=True)
        return arr.rename({variant_dim: variant_axis})

    non_sample_dims = [dim for dim in arr.dims if dim != sample_dim]
    if len(non_sample_dims) == 1:
        return arr.rename({non_sample_dims[0]: variant_axis})
    return arr.stack({variant_axis: non_sample_dims})


def _reduce_to_variant(arr: xr.DataArray) -> xr.DataArray:
    """Average over replicate/technical dimensions, preserving resample and variant."""
    if variant_dim in arr.dims:
        reduce_dims = [dim for dim in arr.dims if dim not in {sample_dim, variant_dim}]
        if reduce_dims:
            arr = arr.mean(dim=reduce_dims, skipna=True)
        return arr.rename({variant_dim: variant_axis})

    if variant_axis in arr.dims:
        reduce_dims = [dim for dim in arr.dims if dim not in {sample_dim, variant_axis}]
        if reduce_dims:
            arr = arr.mean(dim=reduce_dims, skipna=True)
        return arr

    return _as_variant_array(arr)


def reshape_fitness(fitness: xr.DataArray) -> np.ndarray:
    """Returns fitness values as (resample, variant), adding one resample if needed."""
    fitness = _as_variant_array(_to_resamples(fitness))
    if fitness.dims == (variant_axis,):
        return fitness.values[None]

    assert set(fitness.dims) == {variant_axis, sample_dim}, fitness.dims
    return fitness.transpose(sample_dim, variant_axis).values


print(
    f"Using growth_rate_variable={growth_rate_variable!r}: "
    f"MLE dims={idata_mle.mle[growth_rate_variable].dims}, "
    f"HMC analysis dims={_as_variant_array(_to_resamples(idata_hmc.posterior[growth_rate_variable], max_resamples)).dims}"
)


def recalculate_fitness(count_source: str) -> xr.DataArray:
    """
    Recalculates a count-derived fitness proxy from starting/timepoint counts.

    This mirrors the BaseDataAnalyzer dispatch pattern. Chain and draw dimensions
    are stacked into one resample axis before downstream analysis.
    """
    c0, cf = {
        "reported": (reported_starting_counts, reported_timepoint_counts),
        "mle": (
            mle_samples.inference_obj.posterior_predictive.starting_counts,
            mle_samples.inference_obj.posterior_predictive.timepoint_counts,
        ),
        "hmc": (
            hmc_samples.inference_obj.posterior_predictive.starting_counts,
            hmc_samples.inference_obj.posterior_predictive.timepoint_counts,
        ),
    }[count_source]

    if count_source == "mle":
        c0 = _to_resamples(c0)
        cf = _to_resamples(cf)
    elif count_source == "hmc":
        c0 = _to_resamples(c0, max_resamples)
        cf = _to_resamples(cf, max_resamples)

    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="divide by zero encountered in log")
        fitness = np.log((cf + 1) / (c0 + 1))

    fitness = fitness.where(np.isfinite(fitness))
    return _reduce_to_variant(fitness)


def fitness_sanity_check(
    reference_fitness: np.ndarray | pd.Series | xr.DataArray | None = reported_fitness_values,
) -> np.floating | None:
    """Compare count-recalculated fitness to an independent reported-fitness vector."""
    recalculated_fitness = recalculate_fitness("reported").values

    if reference_fitness is None:
        print(
            "Skipping fitness sanity check: observed counts are embedded in the NetCDF, "
            "but no independent reported-fitness vector is loaded."
        )
        return None

    if isinstance(reference_fitness, xr.DataArray):
        reference = reference_fitness.values
    elif isinstance(reference_fitness, pd.Series):
        reference = reference_fitness.to_numpy()
    else:
        reference = np.asarray(reference_fitness)

    if reference.shape != recalculated_fitness.shape:
        raise ValueError(
            "reference_fitness must match recalculated reported fitness shape; "
            f"got {reference.shape}, expected {recalculated_fitness.shape}"
        )

    mask = np.isfinite(reference) & np.isfinite(recalculated_fitness)
    return stats.spearmanr(
        recalculated_fitness[mask],
        reference[mask],
        nan_policy="raise",
    ).statistic


def get_fitness_correlations(
    fitness: xr.DataArray,
    skip_autocorr: bool = False,
) -> np.ndarray | tuple[np.ndarray, np.ndarray]:
    # Flatten the fitness values
    np_fitness = reshape_fitness(fitness)

    # Get the recalculated fitness values
    recalculated_fitness = recalculate_fitness("reported").values
    assert recalculated_fitness.ndim == 1

    # Get spearman correlation between reported and recalculated fitness
    recalculated_vs_sampled = np.array(
        [
            stats.spearmanr(sample, recalculated_fitness, nan_policy="omit").statistic
            for sample in np_fitness
        ]
    )

    # Go no further if there are any NaN values in the provided fitness values
    # and we are not forcing autocorrelation calculation
    if skip_autocorr:
        return recalculated_vs_sampled

    # Otherwise, calculate autocorrelation of the resampled fitness values
    autocorr = (
        faster_autocorrelation(np_fitness)
        if np.isnan(np_fitness).any()
        else stats.spearmanr(np_fitness, axis=1, nan_policy="raise").statistic
    )

    return recalculated_vs_sampled, autocorr


def get_stdev_vs_counts(fitness: xr.DataArray) -> hv.Scatter:
    """Plot standard deviation of fitness values against total counts."""
    reduction_dims = [dim for dim in fitness.dims if dim != variant_axis]
    stdevs = fitness.std(dim=reduction_dims, skipna=True)
    n_nonna = fitness.notnull().sum(dim=reduction_dims)

    total_counts = reported_starting_counts.sum(
        dim=[dim for dim in reported_starting_counts.dims if dim != variant_dim]
    ) + reported_timepoint_counts.sum(
        dim=[dim for dim in reported_timepoint_counts.dims if dim != variant_dim]
    )
    total_counts = _as_variant_array(total_counts)

    return hv.Scatter(
        (total_counts.values, stdevs.values, variants, n_nonna.values),
        kdims=["Total counts"],
        vdims=["Standard deviation", "Variant", "N Fitness Values"],
    ).opts(
        title="Standard Deviation of Fitness Values vs Total Counts",
        xlabel="Total Counts",
        ylabel="Standard Deviation",
        tools=["hover"],
        width=600,
        height=600,
    )


def plot_fitness_distribution(
    fitness: xr.DataArray,
    reference: np.ndarray | None = None,
) -> hv.Overlay:
    """Builds a quantile plot of the fitness values provided by the resamples."""
    fitness_samples = reshape_fitness(fitness)

    if reference is None:
        reference = np.nanmedian(
            fitness_samples, axis=list(range(fitness_samples.ndim - 1))
        )

    if reference.ndim != 1:
        raise ValueError("The reference must be 1D")

    if reference.shape[0] != fitness_samples.shape[-1]:
        raise ValueError("The reference must be the same length as the fitness samples")

    mask = ~np.isnan(reference)
    reference, fitness_samples = reference[mask], fitness_samples[..., mask]

    sorted_indices = np.argsort(reference)
    reference, fitness_samples = (
        reference[sorted_indices],
        fitness_samples[..., sorted_indices],
    )

    ref_rank = stats.rankdata(reference, nan_policy="raise")
    fit_ranks = stats.rankdata(fitness_samples, axis=-1, nan_policy="omit")

    return quantile_plot(
        x=ref_rank,
        reference=fit_ranks,
        quantiles=[0.025, 0.25],
        observed_type="scatter",
        allow_nan=True,
    ).opts(
        title="Quantile Plot of Fitness Values",
        xlabel="Variant Index",
        ylabel="Variant Rank",
        height=600,
        width=600,
    )


def get_growth_rate(source: str) -> xr.DataArray:
    """Returns the growth rate from the specified source and variable."""
    if source == "mle":
        dset = idata_mle.mle if "mle" in idata_mle.groups() else idata_mle.posterior
        base_rate = _to_resamples(dset[growth_rate_variable])
    elif source == "hmc":
        dset = hmc_samples.inference_obj.posterior
        base_rate = _to_resamples(dset[growth_rate_variable], max_resamples)
    else:
        raise ValueError(f"Invalid source: {source}. Must be 'mle' or 'hmc'.")

    base_rate = _as_variant_array(base_rate)

    if growth_rate_variable.startswith("inv_"):
        return 1 / base_rate
    return base_rate


def summarize_correlations(correlations: np.ndarray) -> tuple[float, float]:
    return float(np.nanmean(correlations)), float(np.nanstd(correlations))


def run_analysis(build_plots: bool = False):
    """Runs the notebook version of the flip_data_analysis.py analysis flow."""
    sanity_corr = fitness_sanity_check()
    if sanity_corr is not None:
        print(
            "The correlation between the recalculated and reported fitness values is: "
            f"{sanity_corr:.4f}"
        )

    print("Recalculating fitness values...")
    recalculated_mle = recalculate_fitness("mle")
    recalculated_hmc = recalculate_fitness("hmc")
    r_mle = get_growth_rate("mle")
    assert r_mle.ndim == 1, "Growth rate for MLE should be 1D"
    r_hmc = get_growth_rate("hmc")

    print("Calculating fitness correlations...")
    mle_vs_recalc = get_fitness_correlations(recalculated_mle, skip_autocorr=True)
    hmc_vs_recalc = get_fitness_correlations(recalculated_hmc, skip_autocorr=True)
    r_mle_vs_recalc = get_fitness_correlations(r_mle, skip_autocorr=True)
    r_hmc_vs_recalc, r_hmc_autocorr = get_fitness_correlations(r_hmc)

    if build_plots:
        print("Building quantile plots...")
        quantile_plots = (
            plot_fitness_distribution(recalculated_mle).opts(
                title="MLE: Recalculated Fitness Quantiles"
            )
            + plot_fitness_distribution(recalculated_hmc).opts(
                title="HMC: Recalculated Fitness Quantiles"
            )
            + plot_fitness_distribution(r_hmc).opts(
                title="HMC: Growth Rate Quantiles"
            )
        ).opts(hv.opts.Area(line_width=0.02))

        print("Calculating standard deviation vs counts...")
        stdev_vs_counts = (
            get_stdev_vs_counts(recalculated_mle).opts(
                title="MLE: Recalculated Fitness sigma vs Total Counts"
            )
            + get_stdev_vs_counts(recalculated_hmc).opts(
                title="HMC: Recalculated Fitness sigma vs Total Counts"
            )
            + get_stdev_vs_counts(r_hmc).opts(
                title="HMC: Growth Rate sigma vs Total Counts"
            )
        )
    else:
        quantile_plots = None
        stdev_vs_counts = None

    print("Building dataframe of sampled growth rates...")
    sample_df = pd.DataFrame(
        {
            "variant": variants,
            "MLE": r_mle.values,
            **{
                f"resample_{i}": sample
                for i, sample in enumerate(reshape_fitness(r_hmc))
            },
        }
    )

    print("\nCorrelation summaries (mean, std):")
    for name, values in {
        "MLE Recalculated vs Recalculated Reported": mle_vs_recalc,
        "HMC Recalculated vs Recalculated Reported": hmc_vs_recalc,
        "MLE Growth Rate vs Recalculated Reported": r_mle_vs_recalc,
        "HMC Growth Rate vs Recalculated Reported": r_hmc_vs_recalc,
    }.items():
        mean, std = summarize_correlations(values)
        print(f"  {name}: mean={mean:.4f}, std={std:.4f}")

    return {
        "Correlations": {
            "MLE Recalculated vs Recalculated Reported": mle_vs_recalc,
            "HMC Recalculated vs Recalculated Reported": hmc_vs_recalc,
            "MLE Growth Rate vs Recalculated Reported": r_mle_vs_recalc,
            "HMC Growth Rate vs Recalculated Reported": r_hmc_vs_recalc,
            "HMC Growth Rate Autocorrelation": r_hmc_autocorr,
        },
        "Quantile Plots": quantile_plots,
        "Standard Deviation vs Counts": stdev_vs_counts,
        "Sampled Growth Rates": sample_df,
    }


print("Analysis helpers defined. Run `analysis_results = run_analysis(build_plots=False)` when ready.")

Using growth_rate_variable='log_r_mean': MLE dims=('b',), HMC analysis dims=('a', 'resample')
Analysis helpers defined. Run `analysis_results = run_analysis(build_plots=False)` when ready.


## What the Combined Analysis Is Checking

The goal of this section is to compare three related views of the same experiment: the original observed counts, the MLE/bootstrap fit, and the HMC posterior fit.

First, `recalculate_fitness("reported")` computes a count-derived fitness proxy directly from the observed starting and timepoint counts embedded in the NetCDF file. This is the reference used for the rest of the notebook because it comes from the raw observed counts. If we later load the external reported-fitness CSV used by `flip_data_analysis.py`, `fitness_sanity_check(...)` can compare this count-derived fitness against that independent reported-fitness table. For now that check is skipped rather than replaced with a self-correlation.

Next, `recalculate_fitness("mle")` and `recalculate_fitness("hmc")` do the same count-to-fitness calculation, but using posterior-predictive counts from the MLE/bootstrap and HMC result objects. The main question is whether simulated counts from the fitted model reproduce the same variant-level fitness patterns as the observed counts. A good result is a high and tight distribution of Spearman correlations between each fitted resample and the observed-count recalculation.

The growth-rate comparison asks a slightly different question. The model's direct growth-rate summary, `log_r_mean`, should rank variants similarly to the count-derived fitness proxy. A good fit should show high correlation between growth rate and observed-count fitness, especially for the fitted HMC resamples. This does not need to be exactly one-to-one, but it should preserve the main ordering of variants if the inferred growth-rate parameter is capturing the biological signal.

The quantile plots check whether the rank structure of fitted fitness values is stable across resamples. Ideally, fitted ranks should track the reference ranks smoothly, with narrow quantile bands. Wide bands or strong departures from the diagonal would suggest high uncertainty or systematic mismatch for parts of the variant distribution.

The standard-deviation-versus-counts plots ask whether uncertainty behaves sensibly. Variants with low total counts should generally have higher uncertainty, while variants with many observations should be more stable. A reassuring pattern is lower fitted standard deviation at higher counts, without large high-count outliers.

Finally, the sampled growth-rate dataframe is a convenient export table: one row per variant, the MLE growth-rate estimate, and one column per HMC resample. This is useful for downstream summaries, ranking variants, or passing posterior uncertainty into another analysis.

In [8]:
analysis_results = run_analysis(build_plots=True)

Skipping fitness sanity check: observed counts are embedded in the NetCDF, but no independent reported-fitness vector is loaded.
Recalculating fitness values...
Calculating fitness correlations...
Building quantile plots...
Calculating standard deviation vs counts...
Building dataframe of sampled growth rates...

Correlation summaries (mean, std):
  MLE Recalculated vs Recalculated Reported: mean=0.9236, std=0.0013
  HMC Recalculated vs Recalculated Reported: mean=0.9449, std=0.0008
  MLE Growth Rate vs Recalculated Reported: mean=0.9241, std=0.0000
  HMC Growth Rate vs Recalculated Reported: mean=0.8957, std=0.0014


In [10]:
analysis_results["Quantile Plots"].cols(1) if analysis_results["Quantile Plots"] is not None else None

:Layout
   .Overlay.I   :Overlay
      .Area.I  :Area   [x]   (lower,upper)
      .Area.II :Area   [x]   (lower,upper)
      .Curve.I :Curve   [x]   (y)
   .Overlay.II  :Overlay
      .Area.I  :Area   [x]   (lower,upper)
      .Area.II :Area   [x]   (lower,upper)
      .Curve.I :Curve   [x]   (y)
   .Overlay.III :Overlay
      .Area.I  :Area   [x]   (lower,upper)
      .Area.II :Area   [x]   (lower,upper)
      .Curve.I :Curve   [x]   (y)

In [12]:
analysis_results["Standard Deviation vs Counts"].cols(1) if analysis_results["Standard Deviation vs Counts"] is not None else None

:Layout
   .Scatter.I   :Scatter   [Total counts]   (Standard deviation,Variant,N Fitness Values)
   .Scatter.II  :Scatter   [Total counts]   (Standard deviation,Variant,N Fitness Values)
   .Scatter.III :Scatter   [Total counts]   (Standard deviation,Variant,N Fitness Values)